# 05 — Backtest EMACrossATR

Run the EMA crossover + ATR bracket strategy against BTC-USD-PERP 1h data on a simulated Hyperliquid venue.

**Strategy logic:**
- **Entry:** True EMA crossover (fires once per cross, not every bar in regime)
- **Exit:** ATR-sized bracket orders — market entry + limit TP + stop-market SL
- NT cancels the remaining bracket leg automatically when either TP or SL fills
- EMA reversal while in position: cancel bracket, close via market, wait for next cross
- Default 2:1 reward-to-risk ratio (TP=3.0×ATR, SL=1.5×ATR)

**Sections:**
1. Setup — imports, catalog, engine configuration
2. Single run — fills, positions, account reports
3. Visualisation — price + EMA overlay with trade markers, equity curve
4. Statistics — analyzer performance summary
5. Parameter sweeps — EMA period grid, then ATR multiplier sensitivity

**Prerequisites:** Run `scripts/fetch_hl_candles.py` first to populate `data/catalog/`.

In [ ]:
# ── Cell 1: Imports + shared config ────────────────────────────────
#
# All tuneable values live here. Change once, used everywhere below.

from decimal import Decimal

import pandas as pd
from IPython.display import HTML

from nautilus_trader.analysis import create_tearsheet
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.data import BarType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.persistence.catalog import ParquetDataCatalog

from src.backtesting import make_engine, run_single_backtest
from src.core import bar_type_str
from src.strategies.ema_cross_atr import EMACrossATR, EMACrossATRConfig

from charts import plot_ema_cross, plot_equity_curve, print_summary_stats, plot_pnl_heatmap

# ── Shared config ────────────────────────────────────────────────
CATALOG_PATH     = "../data/catalog"
INSTRUMENT_ID    = "BTC-USD-PERP.HYPERLIQUID"
BAR_INTERVAL     = "1h"
BAR_TYPE_STR     = bar_type_str(INSTRUMENT_ID, BAR_INTERVAL)
VENUE            = Venue("HYPERLIQUID")
STARTING_CAPITAL = 10_000
TRADE_SIZE       = Decimal("0.01")   # 0.01 BTC per trade

# EMA params
FAST_EMA = 20
SLOW_EMA = 50

# ATR bracket params
ATR_PERIOD  = 14
ATR_SL_MULT = 1.5    # Stop-loss = ATR × 1.5
ATR_TP_MULT = 3.0    # Take-profit = ATR × 3.0  (2:1 R:R)

# Sweep grids — EMA periods
FAST_PERIODS = sorted(set([5, 10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100] + [FAST_EMA]))
SLOW_PERIODS = sorted(set([10, 15, 20, 25, 30, 35, 40, 45, 50, 75, 100, 200] + [SLOW_EMA]))

# Sweep grids — ATR multipliers (TP must exceed SL)
ATR_SL_MULTS = [1.0, 1.5, 2.0, 2.5]
ATR_TP_MULTS = [2.0, 3.0, 4.0, 5.0]

print("Imports OK")

In [ ]:
# ── Cell 2: Load data from catalog ────────────────────────────────
catalog    = ParquetDataCatalog(CATALOG_PATH)
instrument = catalog.instruments(instrument_ids=[INSTRUMENT_ID])[0]
bars       = catalog.bars(bar_types=[BAR_TYPE_STR])

print(f"Instrument : {instrument.id}")
print(f"Bar count  : {len(bars):,}")
print(f"First bar  : {pd.Timestamp(bars[0].ts_event, unit='ns', tz='UTC')}")
print(f"Last bar   : {pd.Timestamp(bars[-1].ts_event, unit='ns', tz='UTC')}")

In [ ]:
# ── Cell 3: Configure engine + venue ──────────────────────────────
engine = make_engine(VENUE, instrument, bars, STARTING_CAPITAL)
print("Engine configured.")

In [ ]:
# ── Cell 4: Add EMACrossATR strategy + run ────────────────────────
config = EMACrossATRConfig(
    instrument_id=instrument.id,
    bar_type=BarType.from_str(BAR_TYPE_STR),
    trade_size=TRADE_SIZE,
    fast_ema_period=FAST_EMA,
    slow_ema_period=SLOW_EMA,
    atr_period=ATR_PERIOD,
    atr_sl_multiplier=ATR_SL_MULT,
    atr_tp_multiplier=ATR_TP_MULT,
)
strategy = EMACrossATR(config=config)
engine.add_strategy(strategy)
engine.run()
print("Backtest complete.")

In [ ]:
# ── Cell 5: Reports ───────────────────────────────────────────────
fills_report     = engine.trader.generate_order_fills_report()
positions_report = engine.trader.generate_positions_report()
account_report   = engine.trader.generate_account_report(VENUE)

print(f"Order fills : {len(fills_report)}")
print(f"Positions   : {len(positions_report)}")
print()

print("=== Recent Fills ===")
display(fills_report.tail(10))

print("\n=== Recent Positions ===")
display(positions_report.tail(10))

In [ ]:
# ── Cell 6: Calculate statistics ──────────────────────────────────
#
# Must run before equity curve — analyzer.returns() is a getter that
# only has data after calculate_statistics() populates it.

account   = engine.cache.account_for_venue(VENUE)
positions = engine.cache.position_snapshots() + engine.cache.positions()
analyzer  = engine.portfolio.analyzer
analyzer.calculate_statistics(account, positions)
print(f"Stats calculated — {len(positions)} positions")

In [ ]:
# ── Cell 7: HTML Tearsheet ────────────────────────────────────────
html = create_tearsheet(
    engine,
    output_path=None,
    title=f"EMACrossATR({FAST_EMA}/{SLOW_EMA}, SL={ATR_SL_MULT}x, TP={ATR_TP_MULT}x) | {INSTRUMENT_ID} {BAR_INTERVAL}",
)
display(HTML(html))

In [ ]:
# ── Cell 8: Price chart with EMA overlay + trade markers ──────────
#
# Reuses plot_ema_cross() — same EMA indicators, trade markers show
# bracket entries, TP fills, and SL fills.

fig = plot_ema_cross(
    bars, fills_report,
    fast_period=FAST_EMA,
    slow_period=SLOW_EMA,
    instrument_label=INSTRUMENT_ID,
    bar_label=BAR_INTERVAL,
)
fig.show(config=dict(
    modeBarButtonsToRemove=["autoScale2d", "lasso2d", "select2d"],
    displaylogo=False,
))

In [ ]:
# ── Cell 9: Equity curve ──────────────────────────────────────────
plot_equity_curve(
    analyzer, account_report,
    f"EMACrossATR({FAST_EMA}/{SLOW_EMA}, ATR={ATR_PERIOD}, SL={ATR_SL_MULT}×, TP={ATR_TP_MULT}×)  {INSTRUMENT_ID} {BAR_INTERVAL}",
)

In [ ]:
# ── Cell 10: Summary statistics ────────────────────────────────────
print_summary_stats(analyzer, num_positions=len(positions))

In [ ]:
# ── Cell 11: Parameter sweep — EMA fast/slow periods ─────────────
#
# Grid search over EMA period combinations.
# ATR params held at defaults.

def add_ema_atr_strategy(eng, fast, slow, atr_period=ATR_PERIOD,
                         atr_sl=ATR_SL_MULT, atr_tp=ATR_TP_MULT):
    cfg = EMACrossATRConfig(
        instrument_id=instrument.id,
        bar_type=BarType.from_str(BAR_TYPE_STR),
        trade_size=TRADE_SIZE,
        fast_ema_period=fast,
        slow_ema_period=slow,
        atr_period=atr_period,
        atr_sl_multiplier=atr_sl,
        atr_tp_multiplier=atr_tp,
    )
    eng.add_strategy(EMACrossATR(cfg))

# ── Run the EMA sweep ──
results = []
combos = [(f, s) for f in FAST_PERIODS for s in SLOW_PERIODS if f < s]
total = len(combos)

for i, (fast, slow) in enumerate(combos, 1):
    row = run_single_backtest(
        venue=VENUE, instrument=instrument, bars=bars,
        starting_capital=STARTING_CAPITAL,
        params={"fast": fast, "slow": slow},
        add_strategy=lambda eng, f=fast, s=slow: add_ema_atr_strategy(eng, f, s),
    )
    results.append(row)
    status = f"  !! {row['error']}" if row.get("error") else ""
    print(
        f"  [{i}/{total}] fast={fast:>3}, slow={slow:>3}  "
        f"PnL={row['total_pnl']:>10.2f} PnL%={row['total_pnl_pct']:>7.2f}%"
        f"  positions={row['num_positions']}{status}"
    )

results_df = pd.DataFrame(results)

display_cols = ["fast", "slow", "total_pnl", "total_pnl_pct",
                "num_positions", "final_balance", "min_balance", "error"]
for col in results_df.columns:
    lower = col.lower()
    if any(kw in lower for kw in ["sharpe", "drawdown", "win", "trades", "expectancy", "factor"]):
        display_cols.append(col)

display_cols = [c for c in display_cols if c in results_df.columns]
print("\n=== EMA Parameter Sweep Results ===")
display(results_df[display_cols])

In [ ]:
# ── Cell 12: PnL heatmap — EMA fast vs slow ──────────────────────
plot_pnl_heatmap(
    results_df, row_col="slow", col_col="fast",
    row_label="Slow EMA Period", col_label="Fast EMA Period",
    title="Total PnL (USDC) by EMA Parameters",
)

In [ ]:
# ── Cell 13: ATR multiplier sensitivity sweep ─────────────────────
#
# Fix EMA at the best combo from the previous sweep, vary ATR TP/SL.

best_row = results_df.loc[results_df["total_pnl"].idxmax()]
best_fast = int(best_row["fast"])
best_slow = int(best_row["slow"])
print(f"Best EMA params: fast={best_fast}, slow={best_slow} (PnL={best_row['total_pnl']:,.2f})")
print(f"Sweeping ATR SL={ATR_SL_MULTS} × TP={ATR_TP_MULTS}\n")

atr_results = []
combos = [(sl, tp) for sl in ATR_SL_MULTS for tp in ATR_TP_MULTS if tp > sl]
total = len(combos)

for i, (sl, tp) in enumerate(combos, 1):
    row = run_single_backtest(
        venue=VENUE, instrument=instrument, bars=bars,
        starting_capital=STARTING_CAPITAL,
        params={"atr_sl": sl, "atr_tp": tp},
        add_strategy=lambda eng, _sl=sl, _tp=tp: add_ema_atr_strategy(
            eng, best_fast, best_slow, atr_sl=_sl, atr_tp=_tp,
        ),
    )
    atr_results.append(row)
    status = f"  !! {row['error']}" if row.get("error") else ""
    print(
        f"  [{i}/{total}] SL={sl:.1f}×, TP={tp:.1f}×  "
        f"PnL={row['total_pnl']:>10.2f}  PnL%={row['total_pnl_pct']:>7.2f}%"
        f"  positions={row['num_positions']}{status}"
    )

atr_df = pd.DataFrame(atr_results)

plot_pnl_heatmap(
    atr_df, row_col="atr_sl", col_col="atr_tp",
    row_label="SL Multiplier (×ATR)", col_label="TP Multiplier (×ATR)",
    title=f"Total PnL (USDC) by ATR Multipliers — EMA({best_fast}/{best_slow})",
)

In [ ]:
# ── Cell 14: Cleanup ──────────────────────────────────────────────
engine.dispose()
print("Engine disposed.")